# Synthetic XRF dataset augmentation

This notebook extracts the original background-bank augmentation pipeline from the thesis fine-tuning workflow. It **does not filter any element or layer**. It produces raw and augmented JSON spectra and optionally applies the original 1.4 cm air attenuation used by fine-tuning.

**Inputs:** simulated JSON files containing `counts` and `labels`, plus `example_files/smooth_background_templates.npz`.

**Outputs:** JSON files in `fine_tuning_ood_before_air/train_raw` and `train_aug`, followed by a single `cartella_ft_14cm_ood` directory when air attenuation is enabled.

The original stochastic augmentation settings are preserved; the number of requested samples may be reduced for a quick test.

**Background bank:** this version uses the original `smooth_background_templates.npz` and reads `smooth_mean` from each cluster.


## 1. Paths and augmentation parameters

In [ ]:
from pathlib import Path
from collections import Counter
import json
import os
import shutil
import numpy as np
from glob import glob

# Raw data directory
INPUT_DIRS = [
    Path("example_files/synthetic_json"),
]

#directory of the background templates
BACKGROUND_NPZ = Path("example_files/smooth_background_templates.npz")

#augmented and selected raw data directory
BUILD_ROOT = Path("fine_tuning_ood_before_air")

FINAL_DIR = Path("cartella_ft_14cm_ood")

SEED = 42

#number of total fine tuning spectra
TOTAL_FT = 100000
#fraction of augmented data
FRAC_AUG = 0.85
#Background intensity parameters
S_OVER_B_RANGE = (3.0, 200.0)
ALPHA_RANGE = (0.5, 2.0)

#optional: peak shift
SHIFT_BINS_RANGE = (0, 0)

#target counts for Poisson noise injection
TARGET_PEAK_COUNTS = 1e4

#repetition per raw spectra source file
N_REP_PER_SOURCE = 3
BG_CHOICE_MODE = "per_rep"

#air path length in cm (typically the sample-detector distance)
AIR_THICKNESS_CM = 1.4

APPLY_AIR_ATTENUATION = True
OVERWRITE_OUTPUT = False


## 2. Load and inspect the original experimental background bank

The original `smooth_background_templates.npz` contains three dictionary-like cluster records (`cluster_0`, `cluster_1`, and `cluster_2`). Each record stores its mean background under `smooth_mean`, with 2048 channels. Loading these object arrays requires `allow_pickle=True`; only load files from a trusted source.


In [ ]:
if not BACKGROUND_NPZ.is_file():
    raise FileNotFoundError(f"Background template bank not found: {BACKGROUND_NPZ}")

with np.load(BACKGROUND_NPZ, allow_pickle=True) as data:
    cluster_names = list(data.files)
    clusters = [data[name].item() for name in cluster_names]

bkg_new = np.stack(
    [np.asarray(cluster["smooth_mean"], dtype=np.float32) for cluster in clusters],
    axis=0,
)
assert bkg_new.ndim == 2 and bkg_new.shape[1] == 2048, bkg_new.shape
print("Background templates:", cluster_names, "| shape:", bkg_new.shape)


## 3. Augmentation functions

The helper functions below preserve the original signal scaling, background sampling, Poisson noise, max normalization, labels and metadata.

In [ ]:
def _shift_with_zeros(x, shift_bins: int):
    x = np.asarray(x)
    if shift_bins == 0:
        return x.copy()
    y = np.zeros_like(x)
    if shift_bins > 0:
        y[shift_bins:] = x[:-shift_bins]
    else:
        k = -shift_bins
        y[:-k] = x[k:]
    return y

def _poissonize(x_counts, rng):
    x_counts = np.clip(x_counts, 0.0, None)
    return rng.poisson(x_counts).astype(np.float32)

def _ensure_bg_bank(bg_templates_2048):
    """Convert one or more 2048-channel background templates into a (K, 2048) array."""
    bg = bg_templates_2048
    if isinstance(bg, (list, tuple)):
        bank = [np.asarray(b, dtype=np.float32).ravel() for b in bg]
        if any(b.size != 2048 for b in bank):
            bad = [i for i,b in enumerate(bank) if b.size != 2048][:5]
            raise ValueError(f"Some background are not 2048 long. Example idx={bad}")
        return np.stack(bank, axis=0)
    bg = np.asarray(bg, dtype=np.float32)
    if bg.ndim == 1:
        if bg.size != 2048:
            raise ValueError(f"Background template has to be 2048 bin long, found {bg.size}")
        return bg[None, :]
    if bg.ndim == 2:
        if bg.shape[1] != 2048:
            raise ValueError(f"Background template has to be (K,2048) long, found {bg.shape}")
        return bg
    raise ValueError(f"Background template shape not supported: {bg.shape}")

#raw spectra selection
def make_raw_subset(
    file_paths,
    output_dir,
    n_raw,
    seed=0,
    prefix="raw_",
):
    rng = np.random.default_rng(seed)
    os.makedirs(output_dir, exist_ok=True)

    if len(file_paths) < n_raw:
        raise ValueError(f"Not enough raw file: required {n_raw}, available {len(file_paths)}")

    idx = rng.choice(len(file_paths), size=n_raw, replace=False)
    chosen = [file_paths[i] for i in idx]

    for j, src in enumerate(chosen):
        with open(src, "r") as f:
            d = json.load(f)

        out = {
            "counts": d["counts"],
            "labels": d["labels"],
            "bkg": None,  
            "sim_meta": {
                "finetune_aug": False,
                "source_file": os.path.basename(src),
                "seed": int(seed),
            }
        }

        out_name = f"{prefix}{j:06d}.json"
        with open(os.path.join(output_dir, out_name), "w") as g:
            json.dump(out, g)

    print(f"[make_raw_subset] Created {n_raw} raw files in {output_dir}")
    return chosen

#selected raw spectra augmentation
def make_augmented_from_paths_bg_bank(
    file_paths,
    output_dir,
    bg_templates_2048,
    n_aug_total,
    seed=0,
    s_over_b_range=(10.0, 100.0),
    alpha_range=(0.5, 2.0),
    shift_bins_range=(-8, 8),
    target_peak_counts=1e4,
    save_clean=False,
    prefix="ft_",
    n_rep_per_source=2,
    bg_choice_mode="per_rep",  
):
    rng = np.random.default_rng(seed)
    os.makedirs(output_dir, exist_ok=True)

    bank = _ensure_bg_bank(bg_templates_2048)  
    K = bank.shape[0]
    bank = np.clip(bank, 0.0, None)

    
    n_sources_needed = int(np.ceil(n_aug_total / n_rep_per_source))
    if len(file_paths) < n_sources_needed:
        raise ValueError(f"Not enough source files: {n_sources_needed} required, {len(file_paths)} available")

    
    src_idx = rng.choice(len(file_paths), size=n_sources_needed, replace=False)
    sources = [file_paths[i] for i in src_idx]

    usage = Counter()
    out_idx = 0

    for src_path in sources:
        with open(src_path, "r") as f:
            d = json.load(f)

        counts0 = np.asarray(d["counts"], dtype=np.float32)
        if counts0.shape[0] != 2048:
            if counts0.shape[0] > 2048:
                counts0 = counts0[:2048]
            else:
                counts0 = np.pad(counts0, (0, 2048-counts0.shape[0]), mode="constant", constant_values=0.0)
            counts0 = counts0.astype(np.float32)

        # Background selection mode:
        # "per_source": use the same randomly selected background for all augmented replicas of a source spectrum.
        # "per_rep": independently select a random background for each augmented replica.
        if bg_choice_mode == "per_source":
            bg_idx_src = int(rng.integers(0, K))
        elif bg_choice_mode == "per_rep":
            bg_idx_src = None
        else:
            raise ValueError("bg_choice_mode has to be 'per_rep' o 'per_source'")

        for rep in range(n_rep_per_source):
            if out_idx >= n_aug_total:
                break

            counts = counts0
            s_max = float(np.max(counts))
            if s_max <= 0:
                continue

            
            A = float(target_peak_counts / s_max)
            S_counts = counts * A

            
            bg_idx = bg_idx_src if bg_idx_src is not None else int(rng.integers(0, K))
            bg = bank[bg_idx]
            usage[bg_idx] += 1

            b_max = float(np.max(bg))
            if b_max <= 0:
                B_counts = np.zeros_like(S_counts)
                desired_s_over_b = None
                alpha = None
                sh = 0
                b_scale = 0.0
            else:
                desired_s_over_b = float(rng.uniform(*s_over_b_range))
                b_scale_base = (A * s_max) / (desired_s_over_b * b_max)

                alpha = float(rng.uniform(*alpha_range))
                b_scale = float(b_scale_base * alpha)

                sh = int(rng.integers(shift_bins_range[0], shift_bins_range[1] + 1))
                bg_shift = _shift_with_zeros(bg, sh)

                B_counts = bg_shift * b_scale

            
            X_counts = S_counts + B_counts
            X_poiss = _poissonize(X_counts, rng)

            
            m = float(np.max(X_poiss))
            X_norm = (X_poiss / m).astype(np.float32) if m > 0 else X_poiss.astype(np.float32)

            
            B_norm = (B_counts / m).astype(np.float32) if m > 0 else B_counts.astype(np.float32)

            out = {
                "counts": X_norm.tolist(),
                "labels": d["labels"],
                "bkg": B_norm.tolist(),  
                "sim_meta": {
                    "finetune_aug": True,
                    "source_file": os.path.basename(src_path),
                    "rep": int(rep),
                    "seed": int(seed),
                    "target_peak_counts": float(target_peak_counts),
                    "s_over_b": desired_s_over_b,
                    "alpha": alpha,
                    "shift_bins": int(sh),
                    "b_scale": float(b_scale),
                    "bg_bank_idx": int(bg_idx),
                    "bg_choice_mode": str(bg_choice_mode),
                }
            }
            if save_clean:
                out["counts_clean"] = counts.tolist()

            out_name = f"{prefix}{out_idx:06d}.json"
            with open(os.path.join(output_dir, out_name), "w") as f:
                json.dump(out, f)

            out_idx += 1

    print(f"[make_augmented_from_paths_bg_bank] Created {out_idx} augmented files in {output_dir}")
    return out_idx, sources, usage

## 4. Build the raw and augmented subsets


In [ ]:
all_files = sorted({
    str(path)
    for directory in INPUT_DIRS
    for path in directory.glob("*.json")
})
if not all_files:
    raise FileNotFoundError(
        "No synthetic JSON files found. Place them in example_files/synthetic_json "
        "or update INPUT_DIRS."
    )

n_aug = int(round(TOTAL_FT * FRAC_AUG))
n_raw = TOTAL_FT - n_aug
n_aug_sources = int(np.ceil(n_aug / N_REP_PER_SOURCE))
required_sources = n_raw + n_aug_sources
if len(all_files) < required_sources:
    raise ValueError(
        f"At least {required_sources} distinct source JSON files are needed "
        f"for these settings, but found {len(all_files)}. "
        "Reduce TOTAL_FT for a small example run."
    )

if BUILD_ROOT.exists() and any(BUILD_ROOT.rglob("*.json")):
    if not OVERWRITE_OUTPUT:
        raise FileExistsError(f"{BUILD_ROOT} already contains JSON files.")
    shutil.rmtree(BUILD_ROOT)

rng = np.random.default_rng(SEED)
rng.shuffle(all_files)
raw_sources = all_files[:n_raw]
aug_pool = all_files[n_raw:]

raw_dir = BUILD_ROOT / "train_raw"
aug_dir = BUILD_ROOT / "train_aug"
make_raw_subset(raw_sources, str(raw_dir), n_raw, seed=SEED + 1)
n_written_aug, aug_sources, bg_usage = make_augmented_from_paths_bg_bank(
    file_paths=aug_pool,
    output_dir=str(aug_dir),
    bg_templates_2048=bkg_new,
    n_aug_total=n_aug,
    seed=SEED + 2,
    s_over_b_range=S_OVER_B_RANGE,
    alpha_range=ALPHA_RANGE,
    shift_bins_range=SHIFT_BINS_RANGE,
    target_peak_counts=TARGET_PEAK_COUNTS,
    save_clean=False,
    prefix="ft_",
    n_rep_per_source=N_REP_PER_SOURCE,
    bg_choice_mode=BG_CHOICE_MODE,
)
print("Raw:", len(list(raw_dir.glob("*.json"))))
print("Augmented:", n_written_aug)
print("Background usage:", dict(bg_usage))


## 5. Apply air attenuation

In [ ]:
if APPLY_AIR_ATTENUATION:
    import xraydb
    from tqdm.auto import tqdm
    
    """
    air composition from National Oceanic, Atmospheric Administration, National Aeronautics, Space 
    Administration, and United States Air Force. U.S. Standard Atmosphere, 1976.
    Number NOAA-S/T 76-1562. U.S. Government Printing Office, Washington, D.C., 1976.
    """
    
    AIR_COMP = {"N": 0.755, "O": 0.231, "Ar": 0.013, "C": 0.001}
    RHO_AIR = 0.001225  # g/cm^3

    files_to_process = sorted(raw_dir.glob("*.json")) + sorted(aug_dir.glob("*.json"))
    if not files_to_process:
        raise RuntimeError("No JSON files were generated.")

    with files_to_process[0].open() as handle:
        example = json.load(handle)
    E_eV = np.asarray(
        example.get("E_axis_eV", np.linspace(0.0, 40000.0, 2048)),
        dtype=float,
    )
    if E_eV.shape != (2048,):
        raise ValueError(f"Unexpected energy axis shape: {E_eV.shape}")

    mu_mass = np.zeros_like(E_eV)
    for element, fraction in AIR_COMP.items():
        mu_mass += fraction * np.asarray(
            [xraydb.mu_elam(element, float(E)) for E in E_eV]
        )
    transmission = np.exp(-mu_mass * RHO_AIR * AIR_THICKNESS_CM)

    if FINAL_DIR.exists() and any(FINAL_DIR.glob("*.json")):
        if not OVERWRITE_OUTPUT:
            raise FileExistsError(f"{FINAL_DIR} already contains JSON files.")
        shutil.rmtree(FINAL_DIR)
    FINAL_DIR.mkdir(parents=True, exist_ok=True)

    for path in tqdm(files_to_process, desc="Air attenuation", unit="file"):
        with path.open() as handle:
            record = json.load(handle)
        counts = np.asarray(record["counts"], dtype=float)
        if counts.shape != E_eV.shape:
            raise ValueError(f"Invalid spectrum length: {path}")
        record["counts"] = (counts * transmission).tolist()
        record.setdefault("sim_meta", {})["air_attenuation"] = {
            "thickness_cm": float(AIR_THICKNESS_CM)
        }
        output_name = f"{path.stem}_{path.parent.name}_air14cm.json"
        with (FINAL_DIR / output_name).open("w") as handle:
            json.dump(record, handle)

    print(f"Air-attenuated spectra: {len(files_to_process)} -> {FINAL_DIR}")
else:
    print("Air attenuation disabled. Raw and augmented files remain in", BUILD_ROOT)


## 6. Minimal output validation

In [ ]:
check_dir = FINAL_DIR if APPLY_AIR_ATTENUATION else aug_dir
output_files = sorted(check_dir.glob("*.json"))
if not output_files:
    raise RuntimeError(f"No output JSON files found in {check_dir}")

with output_files[0].open() as handle:
    sample = json.load(handle)

assert len(sample["counts"]) == 2048
assert "labels" in sample
assert "sim_meta" in sample
print("Output directory:", check_dir)
print("Number of JSON files:", len(output_files))
print("First file:", output_files[0].name)
print("Spectrum bins:", len(sample["counts"]))
